In [0]:
%run ../../_shared/00_project_config

In [0]:
%run ../../_shared/01_spark_utils

In [0]:

from functools import reduce

from pyspark.sql import functions as F


# ---------------------------------------------------------------------------
# Source and target tables
# ---------------------------------------------------------------------------

SOURCE_TABLE = f"{GOLD_SCHEMA}.fact_match"

GOLD_TABLE = f"{GOLD_SCHEMA}.fact_team_match"

DIMENSION_TABLES = {
    "team": f"{GOLD_SCHEMA}.dim_team",
    "competition": f"{GOLD_SCHEMA}.dim_competition",
    "season": f"{GOLD_SCHEMA}.dim_season",
    "date": f"{GOLD_SCHEMA}.dim_date",
}


# ---------------------------------------------------------------------------
# Business keys and required columns
# ---------------------------------------------------------------------------

BUSINESS_KEY = [
    "match_key",
    "team_key",
]

REQUIRED_SOURCE_COLUMNS = [
    "match_key",
    "competition_key",
    "season_key",
    "date_key",
    "match_date",
    "home_team_key",
    "away_team_key",
    "match_status",
    "home_goals",
    "away_goals",
    "match_result",
    "home_half_time_goals",
    "away_half_time_goals",
    "home_shots",
    "away_shots",
    "home_shots_on_target",
    "away_shots_on_target",
    "home_corners",
    "away_corners",
    "home_fouls",
    "away_fouls",
    "home_yellow_cards",
    "away_yellow_cards",
    "home_red_cards",
    "away_red_cards",
    "has_csv_source",
    "has_api_source",
]

REQUIRED_GOLD_COLUMNS = [
    "match_key",
    "team_key",
    "opponent_team_key",
    "competition_key",
    "season_key",
    "date_key",
    "match_date",
    "match_status",
    "is_home",
    "home_away",
    "has_csv_source",
    "has_api_source",
]

STATISTIC_COLUMNS = [
    "shots",
    "shots_on_target",
    "corners",
    "fouls",
    "yellow_cards",
    "red_cards",
]


# ---------------------------------------------------------------------------
# Validate source table
# ---------------------------------------------------------------------------

if not spark.catalog.tableExists(SOURCE_TABLE):
    raise ValueError(
        f"Required Gold fact does not exist: {SOURCE_TABLE}"
    )

source_df = spark.table(SOURCE_TABLE)

validate_columns_exist(
    source_df,
    REQUIRED_SOURCE_COLUMNS,
)

source_metrics = validate_required_not_null(
    source_df,
    [
        "match_key",
        "competition_key",
        "season_key",
        "date_key",
        "match_date",
        "home_team_key",
        "away_team_key",
        "match_status",
        "has_csv_source",
        "has_api_source",
    ],
    fail_on_null=True,
)

source_count = source_metrics["row_count"]

if source_count == 0:
    raise ValueError(
        "Source fact_match is empty. "
        "Refusing to build fact_team_match."
    )

duplicate_source_keys = (
    source_df
    .groupBy("match_key")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_source_keys > 0:
    raise ValueError(
        "Duplicate match_key values in fact_match."
    )

invalid_team_pairs = (
    source_df
    .filter(
        F.col("home_team_key") == F.col("away_team_key")
    )
    .limit(1)
    .count()
)

if invalid_team_pairs > 0:
    raise ValueError(
        "A match cannot have identical home and away teams."
    )

print(
    f"Source validated: {SOURCE_TABLE}, "
    f"rows={source_count}"
)


# ---------------------------------------------------------------------------
# Prepare home team perspective
# ---------------------------------------------------------------------------

# Each home record represents the match from the
# perspective of the home team.
#
# Goals for/against and match statistics are renamed
# into team-centric business measures.

home_df = (
    source_df
    .select(
        "match_key",
        "competition_key",
        "season_key",
        "date_key",
        "match_date",
        "match_status",
        "has_csv_source",
        "has_api_source",

        F.col("home_team_key").alias("team_key"),
        F.col("away_team_key").alias("opponent_team_key"),

        F.lit(True).alias("is_home"),
        F.lit("Home").alias("home_away"),

        F.col("home_goals").alias("goals_for"),
        F.col("away_goals").alias("goals_against"),

        F.col("home_half_time_goals")
        .alias("half_time_goals_for"),

        F.col("away_half_time_goals")
        .alias("half_time_goals_against"),

        F.col("home_shots").alias("shots"),
        F.col("home_shots_on_target").alias("shots_on_target"),
        F.col("home_corners").alias("corners"),
        F.col("home_fouls").alias("fouls"),
        F.col("home_yellow_cards").alias("yellow_cards"),
        F.col("home_red_cards").alias("red_cards"),
    )
)


# ---------------------------------------------------------------------------
# Prepare away team perspective
# ---------------------------------------------------------------------------

# Away team metrics are reversed into the same
# team-centric structure used by home teams.
#
# Both DataFrames must have identical column names
# and compatible data types for unionByName.

away_df = (
    source_df
    .select(
        "match_key",
        "competition_key",
        "season_key",
        "date_key",
        "match_date",
        "match_status",
        "has_csv_source",
        "has_api_source",

        F.col("away_team_key").alias("team_key"),
        F.col("home_team_key").alias("opponent_team_key"),

        F.lit(False).alias("is_home"),
        F.lit("Away").alias("home_away"),

        F.col("away_goals").alias("goals_for"),
        F.col("home_goals").alias("goals_against"),

        F.col("away_half_time_goals")
        .alias("half_time_goals_for"),

        F.col("home_half_time_goals")
        .alias("half_time_goals_against"),

        F.col("away_shots").alias("shots"),
        F.col("away_shots_on_target").alias("shots_on_target"),
        F.col("away_corners").alias("corners"),
        F.col("away_fouls").alias("fouls"),
        F.col("away_yellow_cards").alias("yellow_cards"),
        F.col("away_red_cards").alias("red_cards"),
    )
)


# ---------------------------------------------------------------------------
# Combine team perspectives
# ---------------------------------------------------------------------------

# Every match produces two records.
# Future fixtures are retained with null performance metrics.

gold_df = home_df.unionByName(away_df)


# ---------------------------------------------------------------------------
# Derive team-level football metrics
# ---------------------------------------------------------------------------

# Only FINISHED matches contribute goals, results and points.
# Other match statuses remain in the fact for fixture analysis.

finished_condition = (
    F.col("match_status") == "FINISHED"
)

gold_df = (
    gold_df
    .withColumn(
        "goal_difference",
        F.when(
            finished_condition,
            F.col("goals_for") - F.col("goals_against"),
        ).otherwise(F.lit(None).cast("int")),
    )
    .withColumn(
        "match_result",
        F.when(
            ~finished_condition,
            F.lit(None).cast("string"),
        )
        .when(
            F.col("goals_for") > F.col("goals_against"),
            F.lit("W"),
        )
        .when(
            F.col("goals_for") < F.col("goals_against"),
            F.lit("L"),
        )
        .when(
            F.col("goals_for") == F.col("goals_against"),
            F.lit("D"),
        ),
    )
    .withColumn(
        "points_earned",
        F.when(
            F.col("match_result") == "W",
            F.lit(3),
        )
        .when(
            F.col("match_result") == "D",
            F.lit(1),
        )
        .when(
            F.col("match_result") == "L",
            F.lit(0),
        )
        .otherwise(F.lit(None).cast("int")),
    )
    .withColumn(
        "is_win",
        F.when(
            finished_condition,
            F.col("match_result") == "W",
        ).otherwise(F.lit(None).cast("boolean")),
    )
    .withColumn(
        "is_draw",
        F.when(
            finished_condition,
            F.col("match_result") == "D",
        ).otherwise(F.lit(None).cast("boolean")),
    )
    .withColumn(
        "is_loss",
        F.when(
            finished_condition,
            F.col("match_result") == "L",
        ).otherwise(F.lit(None).cast("boolean")),
    )
)


# ---------------------------------------------------------------------------
# Select Gold fact columns
# ---------------------------------------------------------------------------

gold_df = gold_df.select(
    "match_key",
    "team_key",
    "opponent_team_key",
    "competition_key",
    "season_key",
    "date_key",
    "match_date",
    "match_status",
    "is_home",
    "home_away",
    "goals_for",
    "goals_against",
    "goal_difference",
    "half_time_goals_for",
    "half_time_goals_against",
    "match_result",
    "points_earned",
    "is_win",
    "is_draw",
    "is_loss",
    *STATISTIC_COLUMNS,
    "has_csv_source",
    "has_api_source",
)


# ---------------------------------------------------------------------------
# Validate Gold dimensions
# ---------------------------------------------------------------------------

dimension_keys = {
    "team": "team_key",
    "competition": "competition_key",
    "season": "season_key",
    "date": "date_key",
}

dimension_dfs = {}

for dimension_name, table_name in DIMENSION_TABLES.items():

    if not spark.catalog.tableExists(table_name):
        raise ValueError(
            f"Required Gold dimension does not exist: {table_name}"
        )

    df = spark.table(table_name)

    key_column = dimension_keys[dimension_name]

    validate_columns_exist(
        df,
        [key_column],
    )

    validate_required_not_null(
        df,
        [key_column],
        fail_on_null=True,
    )

    duplicates = (
        df.groupBy(key_column)
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )

    if duplicates > 0:
        raise ValueError(
            f"Duplicate keys in Gold dimension: {table_name}"
        )

    dimension_dfs[dimension_name] = (
        df.select(key_column).distinct()
    )

print("Gold dimension keys validated.")


# ---------------------------------------------------------------------------
# Validate foreign key relationships
# ---------------------------------------------------------------------------

foreign_key_checks = [
    ("team_key", "team"),
    ("opponent_team_key", "team"),
    ("competition_key", "competition"),
    ("season_key", "season"),
    ("date_key", "date"),
]

for key_column, dimension_name in foreign_key_checks:

    reference_key = dimension_keys[dimension_name]

    reference_df = (
        dimension_dfs[dimension_name]
        .select(
            F.col(reference_key).alias(key_column)
        )
    )

    missing_keys_df = (
        gold_df
        .select(key_column)
        .distinct()
        .join(
            reference_df,
            on=key_column,
            how="left_anti",
        )
    )

    missing_count = missing_keys_df.count()

    if missing_count > 0:
        display(missing_keys_df)

        raise ValueError(
            f"Found {missing_count} unknown "
            f"{key_column} values."
        )

print("All fact foreign keys reference valid dimensions.")


# ---------------------------------------------------------------------------
# Validate Gold fact grain
# ---------------------------------------------------------------------------

gold_metrics = validate_required_not_null(
    gold_df,
    REQUIRED_GOLD_COLUMNS,
    fail_on_null=True,
)

gold_count = gold_metrics["row_count"]

expected_count = source_count * 2

if gold_count != expected_count:
    raise ValueError(
        "Gold row count mismatch: "
        f"expected={expected_count}, actual={gold_count}"
    )

duplicate_business_keys = (
    gold_df
    .groupBy(*BUSINESS_KEY)
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if duplicate_business_keys > 0:
    raise ValueError(
        "Duplicate match/team business keys detected."
    )

# Each match must produce exactly one home and one away row.

invalid_match_pairs = (
    gold_df
    .groupBy("match_key")
    .agg(
        F.count("*").alias("team_rows"),
        F.sum(
            F.when(F.col("is_home"), 1).otherwise(0)
        ).alias("home_rows"),
        F.sum(
            F.when(~F.col("is_home"), 1).otherwise(0)
        ).alias("away_rows"),
    )
    .filter(
        (F.col("team_rows") != 2)
        | (F.col("home_rows") != 1)
        | (F.col("away_rows") != 1)
    )
)

if invalid_match_pairs.limit(1).count() > 0:
    display(invalid_match_pairs)

    raise ValueError(
        "Each match must contain one home and one away record."
    )

print(
    "Gold fact grain validated: "
    f"{gold_count} team-match records."
)


# ---------------------------------------------------------------------------
# Validate football business rules
# ---------------------------------------------------------------------------

# A team cannot play against itself.

invalid_opponents = (
    gold_df
    .filter(
        F.col("team_key") == F.col("opponent_team_key")
    )
    .limit(1)
    .count()
)

if invalid_opponents > 0:
    raise ValueError(
        "Invalid team/opponent relationship."
    )

# Home and away labels must match is_home.

invalid_home_away = (
    gold_df
    .filter(
        (F.col("is_home") & (F.col("home_away") != "Home"))
        | (
            (~F.col("is_home"))
            & (F.col("home_away") != "Away")
        )
    )
    .limit(1)
    .count()
)

if invalid_home_away > 0:
    raise ValueError(
        "Inconsistent home/away labels."
    )

# Completed matches must have valid results and points.

invalid_finished_results = (
    gold_df
    .filter(
        (F.col("match_status") == "FINISHED")
        & (
            F.col("goals_for").isNull()
            | F.col("goals_against").isNull()
            | F.col("goal_difference").isNull()
            | F.col("match_result").isNull()
            | F.col("points_earned").isNull()
        )
    )
    .limit(1)
    .count()
)

if invalid_finished_results > 0:
    raise ValueError(
        "Finished matches contain missing performance metrics."
    )

# Unfinished fixtures must not contribute results or points.

invalid_unfinished_results = (
    gold_df
    .filter(
        (F.col("match_status") != "FINISHED")
        & (
            F.col("goals_for").isNotNull()
            | F.col("goals_against").isNotNull()
            | F.col("match_result").isNotNull()
            | F.col("points_earned").isNotNull()
        )
    )
    .limit(1)
    .count()
)

if invalid_unfinished_results > 0:
    raise ValueError(
        "Unfinished fixtures contain completed results."
    )

# Result and points must use consistent football rules.

invalid_points = (
    gold_df
    .filter(
        (
            (F.col("match_result") == "W")
            & (F.col("points_earned") != 3)
        )
        | (
            (F.col("match_result") == "D")
            & (F.col("points_earned") != 1)
        )
        | (
            (F.col("match_result") == "L")
            & (F.col("points_earned") != 0)
        )
    )
    .limit(1)
    .count()
)

if invalid_points > 0:
    raise ValueError(
        "Inconsistent points earned and match results."
    )

# Goal difference must equal goals for minus goals against.

invalid_goal_difference = (
    gold_df
    .filter(
        (F.col("match_status") == "FINISHED")
        & (
            F.col("goal_difference")
            != (
                F.col("goals_for")
                - F.col("goals_against")
            )
        )
    )
    .limit(1)
    .count()
)

if invalid_goal_difference > 0:
    raise ValueError(
        "Invalid team-match goal difference."
    )

# Match counts and discipline statistics cannot be negative.

NON_NEGATIVE_COLUMNS = [
    "goals_for",
    "goals_against",
    "shots",
    "shots_on_target",
    "corners",
    "fouls",
    "yellow_cards",
    "red_cards",
]

invalid_counts_condition = reduce(
    lambda left, right: left | right,
    [
        F.col(column_name) < 0
        for column_name in NON_NEGATIVE_COLUMNS
    ],
)

invalid_counts = (
    gold_df
    .filter(invalid_counts_condition)
    .limit(1)
    .count()
)

if invalid_counts > 0:
    raise ValueError(
        "Negative football performance metrics detected."
    )


# ---------------------------------------------------------------------------
# Validate match-level reconciliation
# ---------------------------------------------------------------------------

# Team records must reproduce the source match scores.
# The home team's goals_for must match home_goals.
# The away team's goals_for must match away_goals.

reconstructed_matches_df = (
    gold_df
    .groupBy("match_key")
    .agg(
        F.max(
            F.when(
                F.col("is_home"),
                F.col("goals_for"),
            )
        ).alias("reconstructed_home_goals"),

        F.max(
            F.when(
                ~F.col("is_home"),
                F.col("goals_for"),
            )
        ).alias("reconstructed_away_goals"),
    )
)

source_scores_df = source_df.select(
    "match_key",
    "home_goals",
    "away_goals",
)

score_mismatches = (
    source_scores_df
    .join(
        reconstructed_matches_df,
        on="match_key",
        how="inner",
    )
    .filter(
        ~F.col("home_goals").eqNullSafe(
            F.col("reconstructed_home_goals")
        )
        | ~F.col("away_goals").eqNullSafe(
            F.col("reconstructed_away_goals")
        )
    )
)

score_mismatch_count = score_mismatches.count()

if score_mismatch_count > 0:
    display(score_mismatches)

    raise ValueError(
        f"Found {score_mismatch_count} match score mismatches."
    )

print(
    "Match-level score reconciliation successful."
)


# ---------------------------------------------------------------------------
# Add Gold processing metadata
# ---------------------------------------------------------------------------

gold_df = gold_df.withColumn(
    "_gold_processed_at",
    F.current_timestamp(),
)


# ---------------------------------------------------------------------------
# Write Gold Delta table
# ---------------------------------------------------------------------------

# Full refresh is appropriate for the current project phase.
# Every execution regenerates exactly two team records
# for each source match without appending duplicates.

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}"
)

write_full_refresh(
    gold_df,
    GOLD_TABLE,
)


# ---------------------------------------------------------------------------
# Post-write validation
# ---------------------------------------------------------------------------

saved_df = spark.table(GOLD_TABLE)

saved_metrics = validate_required_not_null(
    saved_df,
    REQUIRED_GOLD_COLUMNS,
    fail_on_null=True,
)

saved_count = saved_metrics["row_count"]

if saved_count != expected_count:
    raise ValueError(
        "Post-write row count mismatch: "
        f"expected={expected_count}, actual={saved_count}"
    )

saved_duplicate_keys = (
    saved_df
    .groupBy(*BUSINESS_KEY)
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if saved_duplicate_keys > 0:
    raise ValueError(
        "Post-write validation failed: "
        "duplicate match/team keys."
    )

saved_invalid_pairs = (
    saved_df
    .groupBy("match_key")
    .agg(
        F.count("*").alias("team_rows"),
        F.sum(
            F.when(F.col("is_home"), 1).otherwise(0)
        ).alias("home_rows"),
        F.sum(
            F.when(~F.col("is_home"), 1).otherwise(0)
        ).alias("away_rows"),
    )
    .filter(
        (F.col("team_rows") != 2)
        | (F.col("home_rows") != 1)
        | (F.col("away_rows") != 1)
    )
    .limit(1)
    .count()
)

if saved_invalid_pairs > 0:
    raise ValueError(
        "Post-write validation failed: "
        "invalid home/away match pairs."
    )

# Verify that no team-match identity was lost during writing.

expected_keys_df = gold_df.select(*BUSINESS_KEY)
saved_keys_df = saved_df.select(*BUSINESS_KEY)

missing_saved_keys = (
    expected_keys_df
    .join(
        saved_keys_df,
        on=BUSINESS_KEY,
        how="left_anti",
    )
    .count()
)

unexpected_saved_keys = (
    saved_keys_df
    .join(
        expected_keys_df,
        on=BUSINESS_KEY,
        how="left_anti",
    )
    .count()
)

if missing_saved_keys > 0 or unexpected_saved_keys > 0:
    raise ValueError(
        "Post-write business keys do not match "
        "the transformed source."
    )


# ---------------------------------------------------------------------------
# Display execution summary
# ---------------------------------------------------------------------------

finished_matches = (
    saved_df
    .filter(F.col("match_status") == "FINISHED")
    .select("match_key")
    .distinct()
    .count()
)

print("=" * 55)
print("GOLD TEAM-MATCH FACT LOAD COMPLETED")
print("=" * 55)
print(f"Target table: {GOLD_TABLE}")
print(f"Source matches: {source_count}")
print(f"Total team-match records: {saved_count}")
print(f"Completed matches: {finished_matches}")
print(f"Duplicate business keys: {saved_duplicate_keys}")
print(f"Score mismatches: {score_mismatch_count}")
print("=" * 55)

display(
    saved_df
    .groupBy(
        "season_key",
        "competition_key",
        "match_status",
    )
    .agg(
        F.count("*").alias("team_match_records"),
        F.countDistinct("match_key").alias("matches"),
    )
    .orderBy(
        "season_key",
        "competition_key",
        "match_status",
    )
)

display(
    saved_df
    .groupBy("home_away", "match_status")
    .agg(
        F.count("*").alias("records"),
        F.sum("points_earned").alias("total_points"),
    )
    .orderBy("home_away", "match_status")
)
